# Comparación de Arquitecturas para Estimación de Biomasa

Experimento completo según la metodología definida en `docs/metodologia_final_justificada.md`.

**Diseño experimental:**
- 6 modelos (ResNet50, EfficientNet-B2, ConvNeXt-Tiny, MaxViT-Tiny, ViT-Small, Swin-Tiny)
- Backbone congelado + cabeza MLP idéntica (128 hidden, dropout 0.3)
- 5-Fold GroupKFold por Sampling_Date
- Targets: log(1+y) + z-score por fold
- Loss: MSE sin ponderación
- Evaluación: Weighted R² (métrica oficial de la competición)

**Modo debug:** Activar `DEBUG = True` para probar el pipeline con pocos epochs (5)
y verificar que MLflow registra correctamente antes de lanzar el experimento completo.

In [ ]:
# =============================================================================
# Celda 1: Setup del entorno (Colab / Local)
# =============================================================================
import os
import sys

# Detectar si estamos en Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Montar Google Drive y cambiar al directorio del proyecto
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    project_path = '/content/drive/MyDrive/image2biomass'
    if os.path.exists(project_path):
        os.chdir(project_path)
        print(f"Directorio de trabajo cambiado a: {os.getcwd()}")
    else:
        print(f"Advertencia: No se encontró el directorio {project_path}")

    # Instalar dependencias necesarias para Databricks + MLflow
    !pip install -q mlflow-skinny databricks-sdk timm

    # Configurar variables de entorno para Databricks
    os.environ["DATABRICKS_HOST"] = ""   # <-- Rellenar con tu host
    os.environ["DATABRICKS_TOKEN"] = ""  # <-- Rellenar con tu token
else:
    # En local, añadir el directorio raíz del proyecto al path
    sys.path.append('../')

print(f"Entorno: {'Google Colab' if IN_COLAB else 'Local'}")

In [ ]:
# =============================================================================
# Celda 2: Imports y configuración del experimento
# =============================================================================
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
import mlflow
import mlflow.pytorch

from utils.paths import get_data_path
from src.config import ExperimentConfig
from src.seed import set_seed
from src.data.dataset import BiomassDataset
from src.data.transforms import get_train_transforms, get_val_transforms
from src.data.preprocessing import TargetScaler, prepare_pivot_table, create_folds
from src.models.factory import create_model, count_parameters
from src.training.trainer import Trainer
from src.evaluation.metrics import evaluate_fold, TARGET_WEIGHTS
from src.inference.predictor import predict_test

# =============================================
# >>> CONFIGURACIÓN PRINCIPAL DEL EXPERIMENTO <<<
# =============================================
# Cambiar a False para el experimento completo (100 epochs)
DEBUG = True

cfg = ExperimentConfig(debug=DEBUG)
set_seed(cfg.seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {device}")
print(f"Modo: {'DEBUG (pocas epochs para probar pipeline)' if DEBUG else 'COMPLETO'}")
print(f"Epochs: {cfg.epochs}")
print(f"Modelos a evaluar: {cfg.model_names}")
print(f"Folds: {cfg.n_folds}")

In [ ]:
# =============================================================================
# Celda 3: Configurar MLflow con Databricks
# =============================================================================
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("")  # <-- Nombre del experimento en Databricks

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

In [ ]:
# =============================================================================
# Celda 4: Cargar datos y preparar tabla pivot
# =============================================================================
base_path = Path(get_data_path())
train_df = pd.read_csv(base_path / 'train.csv')
test_df = pd.read_csv(base_path / 'test.csv')

# Convertir formato largo a ancho (1 fila por imagen)
pivot = prepare_pivot_table(train_df, cfg.targets)

print(f"Imágenes de entrenamiento: {len(pivot)}")
print(f"Targets: {cfg.targets}")
print(f"Columnas del pivot: {list(pivot.columns)}")
print(f"\nEstadísticas de targets (escala original):")
print(pivot[cfg.targets].describe().round(2))
pivot.head()

In [ ]:
# =============================================================================
# Celda 5: Crear folds (GroupKFold por Sampling_Date)
# =============================================================================
folds = create_folds(pivot, n_folds=cfg.n_folds, group_col=cfg.group_col)

print(f"Número de folds: {len(folds)}")
print(f"\nDistribución de muestras por fold:")
for i, (train_idx, val_idx) in enumerate(folds):
    train_fold = pivot.iloc[train_idx]
    val_fold = pivot.iloc[val_idx]
    val_states = val_fold['State'].unique()
    val_dates = val_fold['Sampling_Date'].nunique()
    print(
        f"  Fold {i+1}: train={len(train_idx)}, val={len(val_idx)} | "
        f"Estados val: {sorted(val_states)} | "
        f"Fechas val: {val_dates}"
    )

In [ ]:
# =============================================================================
# Celda 6: Ejecutar experimento completo (todos los modelos × todos los folds)
# =============================================================================
# Aquí se almacenan TODOS los resultados
all_results = []

# Transforms de imagen
train_tfms = get_train_transforms(cfg.img_size)
val_tfms = get_val_transforms(cfg.img_size)

for model_name in cfg.model_names:
    print(f"\n{'=' * 70}")
    print(f"  MODELO: {model_name}")
    print(f"{'=' * 70}")

    # --- Run padre en MLflow (agrupa todos los folds de este modelo) ---
    with mlflow.start_run(run_name=model_name) as parent_run:

        # Log de hiperparámetros comunes
        mlflow.log_params({
            "model_name": model_name,
            "debug_mode": cfg.debug,
            "seed": cfg.seed,
            "img_size": cfg.img_size,
            "batch_size": cfg.batch_size,
            "lr": cfg.lr,
            "epochs": cfg.epochs,
            "early_stopping_patience": cfg.early_stopping_patience,
            "scheduler_patience": cfg.scheduler_patience,
            "scheduler_factor": cfg.scheduler_factor,
            "dropout": cfg.dropout,
            "n_folds": cfg.n_folds,
            "group_col": cfg.group_col,
            "loss": "MSE",
            "optimizer": "Adam",
            "freeze_backbone": True,
            "targets": str(cfg.targets),
        })

        fold_weighted_r2_list = []  # Para agregar al final
        fold_metrics_list = []      # Todos los resultados por fold

        # Guardar el scaler del mejor fold (para submission)
        best_fold_wr2 = -float("inf")
        best_fold_info = None

        for fold_idx, (train_idx, val_idx) in enumerate(folds):
            fold_num = fold_idx + 1
            print(f"\n--- Fold {fold_num}/{cfg.n_folds} ---")

            # --- Run hijo en MLflow (este fold) ---
            with mlflow.start_run(
                run_name=f"{model_name}_fold{fold_num}", nested=True
            ) as child_run:

                mlflow.log_params({
                    "fold": fold_num,
                    "n_train": len(train_idx),
                    "n_val": len(val_idx),
                })

                # 1. Separar train / val
                train_fold_df = pivot.iloc[train_idx].copy()
                val_fold_df = pivot.iloc[val_idx].copy()

                # Guardar verdad en escala original (para evaluar después)
                y_true_original = val_fold_df[cfg.targets].values.copy()

                # 2. Ajustar scaler SOLO con train
                scaler = TargetScaler()
                scaler.fit(train_fold_df[cfg.targets].values)

                # 3. Transformar targets de train y val
                train_fold_df[cfg.targets] = scaler.transform(
                    train_fold_df[cfg.targets].values
                )
                val_fold_transformed = val_fold_df.copy()
                val_fold_transformed[cfg.targets] = scaler.transform(
                    val_fold_df[cfg.targets].values
                )

                # 4. Crear datasets y dataloaders
                train_ds = BiomassDataset(
                    train_fold_df, base_path, cfg.targets, train_tfms
                )
                val_ds = BiomassDataset(
                    val_fold_transformed, base_path, cfg.targets, val_tfms
                )
                train_loader = DataLoader(
                    train_ds,
                    batch_size=cfg.batch_size,
                    shuffle=True,
                    num_workers=cfg.num_workers,
                )
                val_loader = DataLoader(
                    val_ds,
                    batch_size=cfg.batch_size,
                    shuffle=False,
                    num_workers=cfg.num_workers,
                )

                # 5. Crear modelo
                model = create_model(
                    model_name,
                    num_outputs=len(cfg.targets),
                    dropout=cfg.dropout,
                ).to(device)

                # Log de parámetros del modelo (solo en el primer fold)
                if fold_idx == 0:
                    param_info = count_parameters(model)
                    mlflow.log_params(param_info)
                    print(f"  Params entrenables: {param_info['trainable']:,} / {param_info['total']:,}")

                # 6. Entrenar
                criterion = torch.nn.MSELoss()
                optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
                checkpoint_path = (
                    cfg.model_dir / f"{model_name}_fold{fold_num}.pt"
                )

                trainer = Trainer(
                    model, criterion, optimizer, device,
                    scheduler_patience=cfg.scheduler_patience,
                    scheduler_factor=cfg.scheduler_factor,
                )
                history = trainer.fit(
                    train_loader, val_loader,
                    epochs=cfg.epochs,
                    checkpoint_path=checkpoint_path,
                    early_stopping_patience=cfg.early_stopping_patience,
                )

                # Log de curvas de entrenamiento
                for record in history:
                    mlflow.log_metrics(
                        {
                            "train_loss": record["train_loss"],
                            "val_loss": record["val_loss"],
                            "lr": record["lr"],
                        },
                        step=record["epoch"],
                    )

                # 7. Cargar mejor checkpoint
                model.load_state_dict(
                    torch.load(checkpoint_path, map_location=device, weights_only=True)
                )
                model.eval()

                # 8. Obtener predicciones en el val set
                all_preds = []
                with torch.no_grad():
                    for images, _ in val_loader:
                        images = images.to(device)
                        preds = model(images).cpu().numpy()
                        all_preds.append(preds)

                preds_scaled = np.concatenate(all_preds, axis=0)

                # 9. Invertir transformación → escala original
                preds_original = scaler.inverse_transform(preds_scaled)
                preds_original = np.clip(preds_original, 0, None)

                # 10. Evaluar
                fold_result = evaluate_fold(
                    y_true_original, preds_original, cfg.targets
                )

                # Log de métricas del fold
                mlflow.log_metric(
                    "weighted_r2", fold_result["weighted_r2"]
                )
                for target_name, tmetrics in fold_result["per_target"].items():
                    for metric_name, value in tmetrics.items():
                        mlflow.log_metric(
                            f"{target_name}_{metric_name}", value
                        )

                # Guardar info del mejor fold
                if fold_result["weighted_r2"] > best_fold_wr2:
                    best_fold_wr2 = fold_result["weighted_r2"]
                    best_fold_info = {
                        "fold": fold_num,
                        "checkpoint_path": checkpoint_path,
                        "scaler": scaler,
                        "weighted_r2": fold_result["weighted_r2"],
                    }

                fold_weighted_r2_list.append(fold_result["weighted_r2"])
                fold_metrics_list.append(fold_result)

                print(
                    f"  Fold {fold_num} Weighted R²: "
                    f"{fold_result['weighted_r2']:.4f}"
                )

        # --- Métricas agregadas del modelo (media ± std de los folds) ---
        mean_wr2 = float(np.mean(fold_weighted_r2_list))
        std_wr2 = float(np.std(fold_weighted_r2_list))

        mlflow.log_metrics({
            "mean_weighted_r2": mean_wr2,
            "std_weighted_r2": std_wr2,
        })

        # Log de métricas medias por target
        for target_name in TARGET_WEIGHTS.keys():
            for metric_name in ["mae", "rmse", "r2"]:
                values = [
                    fm["per_target"][target_name][metric_name]
                    for fm in fold_metrics_list
                ]
                mlflow.log_metrics({
                    f"mean_{target_name}_{metric_name}": float(np.mean(values)),
                    f"std_{target_name}_{metric_name}": float(np.std(values)),
                })

        print(f"\n  >>> {model_name} | Weighted R²: {mean_wr2:.4f} ± {std_wr2:.4f}")

        # Guardar resultados
        all_results.append({
            "model": model_name,
            "mean_weighted_r2": mean_wr2,
            "std_weighted_r2": std_wr2,
            "fold_wr2_values": fold_weighted_r2_list.copy(),
            "fold_metrics": fold_metrics_list.copy(),
            "best_fold_info": best_fold_info,
        })

print("\n" + "=" * 70)
print("  EXPERIMENTO COMPLETADO")
print("=" * 70)

In [ ]:
# =============================================================================
# Celda 7: Tabla resumen de resultados
# =============================================================================
# Crear tabla con Weighted R² por modelo
summary_rows = []
for result in all_results:
    row = {
        "Modelo": result["model"],
        "Weighted R² (media)": f"{result['mean_weighted_r2']:.4f}",
        "Weighted R² (std)": f"{result['std_weighted_r2']:.4f}",
    }
    # Añadir R² medio por target primario
    for target_name in cfg.targets:
        r2_values = [
            fm["per_target"][target_name]["r2"]
            for fm in result["fold_metrics"]
        ]
        row[f"R² {target_name.replace('Dry_', '').replace('_g', '')}"] = (
            f"{np.mean(r2_values):.4f}"
        )
    # R² de derivados
    for derived in ["GDM_g", "Dry_Total_g"]:
        r2_values = [
            fm["per_target"][derived]["r2"]
            for fm in result["fold_metrics"]
        ]
        row[f"R² {derived.replace('Dry_', '').replace('_g', '')}"] = (
            f"{np.mean(r2_values):.4f}"
        )
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

# Ordenar por Weighted R² descendente
summary_df = summary_df.sort_values(
    "Weighted R² (media)", ascending=False
).reset_index(drop=True)

print("\n=== RANKING DE MODELOS ===")
print(summary_df.to_string(index=False))
summary_df

In [ ]:
# =============================================================================
# Celda 8: Gráficas de comparación
# =============================================================================
plt.style.use("seaborn-v0_8-darkgrid")

# --- Gráfica 1: Weighted R² por modelo (boxplot de folds) ---
fig, ax = plt.subplots(figsize=(10, 5))

model_names_sorted = [r["model"] for r in sorted(
    all_results, key=lambda x: x["mean_weighted_r2"], reverse=True
)]
fold_data = [
    next(r["fold_wr2_values"] for r in all_results if r["model"] == name)
    for name in model_names_sorted
]

ax.boxplot(fold_data, labels=model_names_sorted, patch_artist=True)
ax.set_ylabel("Weighted R²")
ax.set_title("Comparación de Weighted R² por modelo (5-Fold CV)")
ax.tick_params(axis='x', rotation=30)
fig.tight_layout()
plt.show()

# --- Gráfica 2: R² por target para cada modelo ---
all_target_names = list(TARGET_WEIGHTS.keys())
n_targets = len(all_target_names)

fig, axes = plt.subplots(1, n_targets, figsize=(4 * n_targets, 4))

for t_idx, target_name in enumerate(all_target_names):
    ax = axes[t_idx]
    for result in all_results:
        r2_values = [
            fm["per_target"][target_name]["r2"]
            for fm in result["fold_metrics"]
        ]
        mean_r2 = np.mean(r2_values)
        std_r2 = np.std(r2_values)
        ax.barh(
            result["model"], mean_r2,
            xerr=std_r2, capsize=3, alpha=0.7
        )
    ax.set_title(target_name)
    ax.set_xlabel("R²")

fig.suptitle("R² por target y modelo (media ± std)", y=1.02, fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Celda 9: Test estadístico (ANOVA o Kruskal-Wallis)
# =============================================================================
from scipy import stats

# Recopilar Weighted R² de todos los modelos × folds
wr2_by_model = {}
for result in all_results:
    wr2_by_model[result["model"]] = result["fold_wr2_values"]

# Verificar normalidad con Shapiro-Wilk (por modelo)
print("=== Test de normalidad (Shapiro-Wilk) ===")
all_normal = True
for model_name, values in wr2_by_model.items():
    if len(values) >= 3:
        stat, p = stats.shapiro(values)
        normal = p > 0.05
        if not normal:
            all_normal = False
        print(f"  {model_name}: p={p:.4f} {'(normal)' if normal else '(NO normal)'}")

# Test de comparación global
groups = list(wr2_by_model.values())

if all_normal:
    print("\n=== ANOVA (datos normales) ===")
    stat, p_value = stats.f_oneway(*groups)
    test_name = "ANOVA"
else:
    print("\n=== Kruskal-Wallis (datos NO normales) ===")
    stat, p_value = stats.kruskal(*groups)
    test_name = "Kruskal-Wallis"

print(f"  Estadístico: {stat:.4f}")
print(f"  p-value: {p_value:.4f}")
print(f"  Significativo (α=0.05): {'Sí' if p_value < 0.05 else 'No'}")

# Post-hoc si es significativo
if p_value < 0.05:
    print(f"\n=== Post-hoc: comparaciones por pares ===")
    model_names = list(wr2_by_model.keys())
    for i in range(len(model_names)):
        for j in range(i + 1, len(model_names)):
            name_i = model_names[i]
            name_j = model_names[j]
            stat_pair, p_pair = stats.mannwhitneyu(
                wr2_by_model[name_i],
                wr2_by_model[name_j],
                alternative='two-sided',
            )
            sig = "*" if p_pair < 0.05 else "ns"
            print(f"  {name_i} vs {name_j}: p={p_pair:.4f} {sig}")
else:
    print("\n  No se encontraron diferencias significativas entre modelos.")

In [ ]:
# =============================================================================
# Celda 10: Generar submission con el mejor modelo
# =============================================================================
# Encontrar el mejor modelo global
best_result = max(all_results, key=lambda r: r["mean_weighted_r2"])
best_model_name = best_result["model"]
best_info = best_result["best_fold_info"]

print(f"Mejor modelo: {best_model_name}")
print(f"  Mean Weighted R²: {best_result['mean_weighted_r2']:.4f}")
print(f"  Mejor fold: {best_info['fold']} (Weighted R²: {best_info['weighted_r2']:.4f})")

# Cargar el mejor modelo
best_model = create_model(
    best_model_name,
    num_outputs=len(cfg.targets),
    dropout=cfg.dropout,
).to(device)

best_model.load_state_dict(
    torch.load(best_info["checkpoint_path"], map_location=device, weights_only=True)
)
best_model.eval()

# Generar submission
submission = predict_test(
    model=best_model,
    test_df=test_df,
    targets=cfg.targets,
    scaler=best_info["scaler"],
    images_root=base_path,
    transform=val_tfms,
    device=device,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
)

# Guardar submission
os.makedirs(cfg.model_dir, exist_ok=True)
submission_path = cfg.model_dir / f"submission_{best_model_name}.csv"
submission.to_csv(submission_path, index=False)

print(f"\nSubmission guardado en: {submission_path}")
print(f"Filas: {len(submission)}")
submission